In [23]:
# Standard Data Package
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Datetime Package
from datetime import datetime

# Serialization Package
import joblib

In [3]:
data = pd.read_excel("../data/raw/online_retail_II.xlsx")

In [4]:
# Data Preview
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
# Checking Data Info
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB


- There seems to be some missing value on `Customer ID` and `Description`
- `Customer ID` should be an object rather than float64

In [6]:
# Checking Missing Values
data[data['Customer ID'].isna()]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
525231,538159,21324,NaN,-18,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525232,538158,20892,NaN,-32,2010-12-09 17:17:00,0.00,NaN,United Kingdom
525233,538160,20956,NaN,288,2010-12-09 17:18:00,0.00,NaN,United Kingdom
525234,538161,46000S,Dotcom sales,-100,2010-12-09 17:25:00,0.00,NaN,United Kingdom


- Since `Customer ID` is a customer identifier, transactions without `Customer ID` will be dropped.

In [7]:
data = data.dropna(subset='Customer ID')

#Validation
print('Number of missing `Customer ID` rows: ', data['Customer ID'].isna().sum())

Number of missing `Customer ID` rows:  0


In [8]:
data['Customer ID'] = data['Customer ID'].astype(int).astype(str)

#Validation
print('`Customer ID` data type: ', data['Customer ID'].dtype)

`Customer ID` data type:  object


In [9]:
# Check for Duplicates
print('Number of duplicated rows:', data.duplicated(keep = 'first').sum())

data[data.duplicated(keep = 'first')]

Number of duplicated rows: 6771


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
383,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329,United Kingdom
384,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329,United Kingdom
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329,United Kingdom
...,...,...,...,...,...,...,...,...
523258,538035,20728,LUNCH BAG CARS BLUE,3,2010-12-09 13:03:00,1.65,16065,United Kingdom
523356,538051,22659,LUNCH BOX I LOVE LONDON,2,2010-12-09 13:24:00,1.95,18041,United Kingdom
525170,538155,21907,I'M ON HOLIDAY METAL SIGN,1,2010-12-09 16:52:00,2.10,16907,United Kingdom
525293,538166,21915,RED HARMONICA IN BOX,5,2010-12-09 18:09:00,1.25,17965,United Kingdom


In [10]:
data = data.drop_duplicates(keep ='first')

#Validation
print('Number of duplicated rows:', data.duplicated(keep = 'first').sum())

Number of duplicated rows: 0


In [11]:
# Checking Descriptive Stats

# Numeric Columns
data.describe()

,Quantity,InvoiceDate,Price
count,410763.000000,410763,410763.000000
mean,12.923735,2010-06-30 19:56:14.853674752,3.908358
min,-9360.000000,2009-12-01 07:45:00,0.000000
25%,2.000000,2010-03-26 09:46:00,1.250000
50%,5.000000,2010-07-08 15:09:00,1.950000
75%,12.000000,2010-10-14 12:32:00,3.750000
max,19152.000000,2010-12-09 20:01:00,25111.090000
std,102.039550,NaN,71.714794


- `Quantity` has negative values when it should be positive.
- `Price` has 0 as a value.
- Both `Quantity` and `Price` have outliers.
- This dataset only contains data from  1 December 2009 to 9 December 2010.     

In [12]:
# Checking negative value on `Quantity`
data[data['Quantity']<1]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321,Australia
...,...,...,...,...,...,...,...,...
524695,C538123,22956,36 FOIL HEART CAKE CASES,-2,2010-12-09 15:41:00,2.10,12605,Germany
524696,C538124,M,Manual,-4,2010-12-09 15:43:00,0.50,15329,United Kingdom
524697,C538124,22699,ROSES REGENCY TEACUP AND SAUCER,-1,2010-12-09 15:43:00,2.95,15329,United Kingdom
524698,C538124,22423,REGENCY CAKESTAND 3 TIER,-1,2010-12-09 15:43:00,12.75,15329,United Kingdom


- According to a discussion about the dataset in the kaggle community, it seems that the negative valued `Quantity` is a canceled transaction and have its corresponding `Invoice` started with C.
- Let's confirm if this is the case.

In [13]:
data['Invoice'] = data['Invoice'].astype(str)

neg_qt =data[data['Quantity']< 1]
c_invoice = data[data['Invoice'].str.startswith('C')]
c_and_neg= data[(data['Invoice'].str.startswith('C') )& (data['Quantity']< 1)]

print("Negative Quantity data shape: ", neg_qt.shape)
print("C Invoice data shape: ", c_invoice.shape)
print("C Invoice and Negative Quantity data shape: ",c_and_neg.shape)

Negative Quantity data shape:  (9816, 8)
C Invoice data shape:  (9816, 8)
C Invoice and Negative Quantity data shape:  (9816, 8)


- It's confirmed that all the negative valued `Quantity` have it's corresponding `Invoice` started with C.
- Let's drop all the cancelled transactions.

In [14]:
#Checking if `Quantity`has 0 as its value
data[data['Quantity']==0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country


In [15]:
#Dropping cancelled transactions
print('Data shape before dropping cancelled transactions: ', data.shape)
data = data[data['Quantity']>=1]
print('Data shape after dropping cancelled transactions: ', data.shape)

Data shape before dropping cancelled transactions:  (410763, 8)
Data shape after dropping cancelled transactions:  (400947, 8)


In [16]:
data['Quantity'].describe()

count    400947.000000
mean         13.768523
std          97.639816
min           1.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       19152.000000
Name: Quantity, dtype: float64

In [17]:
# Checking outliers on `Quantity`
Q1 = data['Quantity'].quantile(0.25)
Q3 = data['Quantity'].quantile(0.75)
IQR = Q3 - Q1
upper_whisker = Q3 + 1.5*IQR
lower_whisker = Q1 + 1.5*IQR

pd.set_option('display.max_rows', None)
data[data['Quantity']> upper_whisker].sort_values(by='Quantity', ascending=False).head(30)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902,Denmark
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902,Denmark
135027,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135029,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
135030,502269,21981,PACK OF 12 WOODLAND TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom
93677,498152,85220,SMALL FAIRY CAKE FRIDGE MAGNETS,9456,2010-02-17 10:51:00,0.30,13902,Denmark


- Let's calculate the total price and see if it's still reasonable

In [18]:
# Creating `Total_Price` column
data['Total_Price'] = data['Quantity'] * data['Price']

# `Total_Price` descriptive statistics
data[data['Quantity']> upper_whisker]['Total_Price'].describe()

count    26296.000000
mean       123.941701
std        241.398509
min          0.000000
25%         30.600000
50%         69.600000
75%        136.000000
max      15818.400000
Name: Total_Price, dtype: float64

In [19]:
data[data['Total_Price']> 140].sort_values(by='Total_Price', ascending=False).head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_Price
432176,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838,United Kingdom,15818.40
135013,502263,M,Manual,1,2010-03-23 15:22:00,10953.50,12918,United Kingdom,10953.50
358639,524159,M,Manual,1,2010-09-27 16:12:00,10468.80,14063,United Kingdom,10468.80
74356,496115,M,Manual,1,2010-01-29 11:04:00,8985.60,17949,United Kingdom,8985.60
228042,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008,United Kingdom,8925.00
129987,501768,M,Manual,1,2010-03-19 11:45:00,6958.17,15760,Norway,6958.17
129903,501766,M,Manual,1,2010-03-19 11:35:00,6958.17,15760,Norway,6958.17
379875,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,2010-10-08 10:10:00,1.66,15838,United Kingdom,5179.20
358821,524181,21622,VINTAGE UNION JACK CUSHION COVER,648,2010-09-27 16:59:00,6.89,17450,United Kingdom,4464.72
452200,532358,84879,ASSORTED COLOUR BIRD ORNAMENT,2880,2010-11-11 17:05:00,1.45,12931,United Kingdom,4176.00


- The `Total_Price` still seems to be reasonable for a normal transaction.

In [20]:
# Checking rows with 0 price
display(data[data['Price'] == 0])
print('0 `Price` Data shape: ', data[data['Price'] ==0].shape)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Total_Price
4674,489825,22076,6 RIBBONS EMPIRE,12,2009-12-02 13:34:00,0.0,16126,United Kingdom,0.0
6781,489998,48185,DOOR MAT FAIRY CAKE,2,2009-12-03 11:19:00,0.0,15658,United Kingdom,0.0
16107,490727,M,Manual,1,2009-12-07 16:38:00,0.0,17231,United Kingdom,0.0
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,2009-12-08 15:25:00,0.0,14108,United Kingdom,0.0
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,2009-12-08 15:25:00,0.0,14108,United Kingdom,0.0
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,2009-12-15 13:49:00,0.0,15070,United Kingdom,0.0
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,2009-12-18 14:22:00,0.0,18071,United Kingdom,0.0
47126,493761,79320,FLAMINGO LIGHTS,24,2010-01-06 14:54:00,0.0,14258,United Kingdom,0.0
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,2010-01-08 10:43:00,0.0,12417,Belgium,0.0
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,2010-01-15 12:43:00,0.0,16858,United Kingdom,0.0


0 `Price` Data shape:  (31, 9)


- It's likely to be an input error but it's still a valid transaction data.

In [21]:
#Categorical Columns
data.describe(include='object').T

,count,unique,top,freq
Invoice,400947,19215,500356,251
StockCode,400947,4017,85123A,3107
Description,400947,4444,WHITE HANGING HEART T-LIGHT HOLDER,3107
Customer ID,400947,4314,14911,5568
Country,400947,37,United Kingdom,364255


In [22]:
joblib.dump(data, "../data/processed/cleaned_data.pkl")

['../data/processed/cleaned_data.pkl']